# Notebook 12 — Streamlit demonstration app (for the Appendix A screenshots)  ·  GPU: T4

Runs the updated demo on Colab's GPU and gives you a public link to open in your laptop's browser; your laptop only displays the page.

The app uses the system described in the paper: Llama 3 8B fine-tuned **with counter-examples** (fold 0 model), the same prompt and probability-based scoring, and the lexical components. It also shows the standard fine-tuned model for comparison. The Table A1 answer belongs to fold 0, so the app should reproduce the Table A1 scores (tiny differences in the second decimal are possible).

**Run all cells.** The last cell prints a link ending in `trycloudflare.com`. Open it, wait ~2–3 min for the models to load the first time, then paste in the Table A1 question, reference answer and student answers.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name --format=csv

Mounted at /content/drive
name
Tesla T4


In [ ]:
!pip -q install -U streamlit bitsandbytes accelerate peft sentence-transformers language-tool-python
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 114.3 MB/s eta 0:00:00


In [ ]:
%%writefile /content/app.py
import sys, numpy as np, pandas as pd, streamlit as st, nltk
ROOT = "/content/drive/MyDrive/asag_revision"
sys.path.insert(0, ROOT)
for p in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(p, quiet=True)
from asag_utils import keyword_score, synonym_score, grammar_score, context_score, composite_fixed
from llm_utils import BASE_MODEL, SCORE_TARGETS, load_hf_4bit, candidate_ids, candidate_logprobs, score_from_logprobs, make_prompts

FOLD = 0      # the Table A1 example belongs to test fold 0
st.set_page_config(page_title="Short-answer grading demo", layout="centered")

@st.cache_resource(show_spinner="Loading Llama 3 8B and the fine-tuned adapters (first time only, ~2-3 min)...")
def load_llm():
    from peft import PeftModel
    model, tok = load_hf_4bit(BASE_MODEL)
    model = PeftModel.from_pretrained(model, f"{ROOT}/models/mohler_llama3_ftaug_f{FOLD}", adapter_name="ftaug")
    model.load_adapter(f"{ROOT}/models/mohler_llama3_ft_f{FOLD}", adapter_name="ft")
    model.eval()
    cand, _ = candidate_ids(tok, SCORE_TARGETS)
    return model, tok, cand

@st.cache_resource(show_spinner="Loading the lexical module...")
def load_lexical():
    import language_tool_python
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LinearRegression
    mo = pd.read_csv(f"{ROOT}/data/mohler_prepared.csv")
    feats = pd.read_csv(f"{ROOT}/cache/features.csv")
    fit = mo.merge(feats, on="uid")
    fit = fit[fit[f"split_f{FOLD}"] != "test"]
    rescale = LinearRegression().fit(fit[["composite_fixed"]], fit.y)      # as in Notebook 0
    return (language_tool_python.LanguageTool("en-US"),
            SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"), rescale)

def llm_score(q, r, s, adapter):
    model, tok, cand = load_llm()
    model.set_adapter(adapter)
    df = pd.DataFrame({"question": [q], "reference_answer": [r], "student_answer": [s]})
    lp = candidate_logprobs(model, tok, make_prompts(df, "mohler"), cand, batch_size=1)
    return float(score_from_logprobs(lp)[0][0])

def lexical_scores(r, s):
    tool, st_model, rescale = load_lexical()
    e = st_model.encode([r, s], normalize_embeddings=True)
    comp = {"keyword": keyword_score(r, s), "synonym": synonym_score(r, s),
            "grammar": grammar_score(len(tool.check(s))), "context": float(context_score(float(e[0] @ e[1])))}
    raw = float(composite_fixed(pd.DataFrame([comp]))[0])
    comp["composite"] = float(np.clip(rescale.predict(pd.DataFrame({"composite_fixed": [raw]}))[0], 0, 10))
    return comp

load_llm(); load_lexical()     # load once at start-up

st.title("Short-Answer Grading Demo")
st.caption("Research prototype. Scores should be reviewed by a teacher before use in assessment.")
q = st.text_area("Question", height=80)
r = st.text_area("Reference answer", height=110)
s = st.text_area("Student answer", height=110)

if st.button("Evaluate answer", type="primary") and q.strip() and r.strip() and s.strip():
    with st.spinner("Scoring..."):
        grade = llm_score(q, r, s, "ftaug")
        grade_ft = llm_score(q, r, s, "ft")
        lex = lexical_scores(r, s)
    st.subheader("Evaluation")
    st.metric("Grade (fine-tuned Llama 3 8B with counter-examples)", f"{grade:.1f} / 10")
    st.markdown("**Lexical–semantic module**")
    c = st.columns(5)
    for col, (name, key) in zip(c, [("Keyword", "keyword"), ("Synonym", "synonym"), ("Grammar", "grammar"),
                                    ("MiniLM context", "context"), ("Composite (rescaled)", "composite")]):
        col.metric(name, f"{lex[key]:.1f}")
    with st.expander("Comparison: standard fine-tuned model (without counter-examples)"):
        st.metric("Grade (standard fine-tuned Llama 3 8B)", f"{grade_ft:.1f} / 10")

Writing /content/app.py


In [ ]:
import subprocess, time, re
subprocess.Popen('streamlit run /content/app.py --server.port 8501 --server.headless true > /content/st.log 2>&1', shell=True)
time.sleep(5)
subprocess.Popen('/content/cloudflared tunnel --url http://localhost:8501 > /content/cf.log 2>&1', shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', open('/content/cf.log').read())
    if m: url = m.group(0); break
print('Open this link in your browser:', url or 'not ready yet - re-run this cell')

Open this link in your browser: https://pill-rocket-incidence-glasses.trycloudflare.com


If the page shows an error, run the next cell and send me its output. Keep this notebook open while you use the app; closing the session stops it.

In [ ]:
!tail -n 30 /content/st.log



2026-10-14 13:10:34.275 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.103.239:8501

